## Silver To Gold


#### Apply fraud detection rules and transform Silver data into business-ready Gold tables

Run the Configurations file

In [0]:
%run "/Workspace/bank-project/00-Configurations"

Import necessary functions, libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
import logging

Silver to Gold Logic

In [0]:
class SilverToGold:

    def __init__(self, spark, silver_table, gold_base_path):
        self.spark = spark
        self.silver_table = silver_table
        self.gold_base_path = gold_base_path

        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger("SilverToGoldPipeline")

        self.pipeline_name = "silver_to_gold_pipeline"

        # Rule thresholds
        self.HIGH_VALUE_THRESHOLD = 1000
        self.TIME_WINDOW_MINUTES = 10
        self.MULTI_TXN_THRESHOLD = 3

    def read_silver(self):
        self.logger.info("Reading silver table")
        df = self.spark.read.table(self.silver_table)
        return df

    def apply_rules(self, df):
        self.logger.info("Applying fraud rules")

        # Rule 1: High Value
        df = df.withColumn(
            "rule_high_value",
            when(col("transaction_amount") > self.HIGH_VALUE_THRESHOLD, 1).otherwise(0)
        )

        # Rule 2: Multiple transactions in short time
        window_spec = Window.partitionBy("card_number").orderBy(col("transaction_time").cast("long"))

        df = df.withColumn(
            "txn_count_10min",
            count("*").over(window_spec.rangeBetween(-self.TIME_WINDOW_MINUTES * 60, 0))
        )

        df = df.withColumn(
            "rule_multiple_txn",
            when(col("txn_count_10min") > self.MULTI_TXN_THRESHOLD, 1).otherwise(0)
        )

        return df

    def create_fraud_columns(self, df):
        self.logger.info("Creating fraud flag and reason")

        df = df.withColumn(
            "is_fraud",
            when((col("rule_high_value") == 1) | (col("rule_multiple_txn") == 1), 1).otherwise(0)
        )

        df = df.withColumn(
            "fraud_reason",
            when(col("rule_high_value") == 1, "High Value Transaction")
            .when(col("rule_multiple_txn") == 1, "Multiple Transactions in Short Time")
            .otherwise("Normal")
        )

        return df

    def create_gold_tables(self, df):
        self.logger.info("Creating gold dataframes")

        fraud_df = df\
            .filter(
                (col("rule_high_value") == 1) |
                (col("rule_multiple_txn") == 1) 
                )\
            .select(
                "transaction_number","transaction_time","merchant","category","transaction_amount","card_number","merchant_latitude","merchant_longitude","merchant_zipcode","fraud_reason"
            )

        customer_df = df.select(
            "card_number","first_name","last_name","gender","street","city","state","zipcode","latitude","longitude","job","dob"
        ).dropDuplicates()

        transaction_df = df.select(
            "transaction_number","transaction_time","merchant","category","transaction_amount","card_number","first_name","last_name","gender","street","city","state","zipcode","latitude","longitude","job","dob","city_population","merchant_latitude","merchant_longitude","merchant_zipcode","is_fraud","fraud_reason"
        )

        return fraud_df, customer_df, transaction_df

    def write_gold(self, fraud_df, customer_df, transaction_df):
        self.logger.info("Writing gold tables")

        fraud_path = f"{self.gold_base_path}/fact_fraud_transactions"
        customer_path = f"{self.gold_base_path}/dim_customer_details"
        transaction_path = f"{self.gold_base_path}/fact_all_transactions"

        fraud_df.write.format("delta").mode("overwrite") \
            .option("path", fraud_path) \
            .saveAsTable("cc_trans_catalog.trans_gold.fact_fraud_transactions")

        customer_df.write.format("delta").mode("overwrite") \
            .option("path", customer_path) \
            .saveAsTable("cc_trans_catalog.trans_gold.dim_customer_details")

        transaction_df.write.format("delta").mode("overwrite") \
            .option("path", transaction_path) \
            .saveAsTable("cc_trans_catalog.trans_gold.fact_all_transactions")

    def log_to_table(self, step, status, record_count=0, error_msg=None):

        if error_msg is None:
            error_msg = ""

        schema = StructType([
            StructField("pipeline_name", StringType(), True),
            StructField("step_name", StringType(), True),
            StructField("status", StringType(), True),
            StructField("record_count", LongType(), True),
            StructField("error_message", StringType(), True)
        ])

        log_data = [(
            self.pipeline_name,
            step,
            status,
            record_count,
            error_msg
        )]

        log_df = self.spark.createDataFrame(log_data, schema) \
            .withColumn("log_time", current_timestamp())

        log_df.write.mode("append") \
            .saveAsTable("cc_trans_catalog.audit.pipeline_logs")

        self.logger.info(f"{step} - {status}")

    def run_pipeline(self):
        try:
            df = self.read_silver()
            self.log_to_table("read_silver", "SUCCESS", df.count())

            df = self.apply_rules(df)
            self.log_to_table("apply_rules", "SUCCESS")

            df = self.create_fraud_columns(df)
            self.log_to_table("create_fraud_columns", "SUCCESS")

            fraud_df, customer_df, transaction_df = self.create_gold_tables(df)
            self.log_to_table("create_gold_tables", "SUCCESS", fraud_df.count())

            self.write_gold(fraud_df, customer_df, transaction_df)
            self.log_to_table("write_gold", "SUCCESS", fraud_df.count())

            self.logger.info("Pipeline completed successfully")
            return df

        except Exception as e:
            self.log_to_table("pipeline_failed", "FAILED", 0, str(e))
            self.logger.error(str(e))
            raise

Object creation

In [0]:
pipeline = SilverToGold(
    spark=spark,
    silver_table="cc_trans_catalog.trans_silver.cc_transactions_silver",
    gold_base_path=gold_path
)



Execute Pipeline

In [0]:
pipeline.run_pipeline()

INFO:SilverToGoldPipeline:Reading silver table
INFO:SilverToGoldPipeline:read_silver - SUCCESS
INFO:SilverToGoldPipeline:Applying fraud rules
INFO:SilverToGoldPipeline:apply_rules - SUCCESS
INFO:SilverToGoldPipeline:Creating fraud flag and reason
INFO:SilverToGoldPipeline:create_fraud_columns - SUCCESS
INFO:SilverToGoldPipeline:Creating gold dataframes
INFO:SilverToGoldPipeline:create_gold_tables - SUCCESS
INFO:SilverToGoldPipeline:Writing gold tables
INFO:SilverToGoldPipeline:write_gold - SUCCESS
INFO:SilverToGoldPipeline:Pipeline completed successfully


DataFrame[transaction_number: string, transaction_time: timestamp, card_number: string, merchant: string, category: string, transaction_amount: double, first_name: string, last_name: string, gender: string, street: string, city: string, state: string, zipcode: int, latitude: double, longitude: double, city_population: int, job: string, dob: date, merchant_latitude: double, merchant_longitude: double, merchant_zipcode: int, ingestion_time: timestamp, rule_high_value: int, txn_count_10min: bigint, rule_multiple_txn: int, is_fraud: int, fraud_reason: string]